# Option Pricing and Greeks Analysis

## Introduction

This notebook demonstrates how to use QuantStrata for:
1. **Pricing options** using the Black-Scholes-Merton model
2. **Computing Greeks** (delta, gamma, vega, theta, rho)
3. **Visualizing** option payoffs and sensitivities

---

### What You'll Learn

- How to set up realistic market data (spot, term structures, vol surfaces)
- How to create option instruments
- How to price options and compute Greeks
- How to interpret Greek values for risk management
- How to visualize P&L profiles and sensitivities

---

### Prerequisites

- QuantStrata library installed (`pip install -e .`)
- Python 3.12+
- matplotlib for plots

In [ ]:
# =============================================================================
# SETUP: Imports and Configuration
# =============================================================================

# Standard library
import sys
from pathlib import Path
from datetime import date
import numpy as np

# Ensure src is importable (adjust path if needed)
sys.path.insert(0, str(Path.cwd().parents[1]))

# Plotting
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')  # Clean, professional style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# QuantStrata imports
from src.marketdata.core.market import Market
from src.marketdata.core.ids import MarketId
from src.marketdata.core.interfaces import Quote
from src.marketdata.curves.term_structure import ZeroRateCurve
from src.marketdata.surfaces.vol_surface import GridVolSurface
from src.instruments.fx.options.vanilla import FxVanillaEuropeanOption
from src.pricers.fx.european_bsm import FxVanillaEuropeanOptionBsmPricer

print("All imports successful!")

## 1. Setting Up Realistic Market Data

Before we can price options, we need to set up the market environment with **realistic** data:

| Component | Description | Real-World Source |
|-----------|-------------|-------------------|
| **Spot Quote** | Current FX rate | Bloomberg/Reuters |
| **Zero Rate Curve** | Term structure of interest rates | Bootstrapped from swap rates |
| **Vol Surface** | Implied vol by expiry and strike | Option market quotes |

### Why Term Structures Matter

- **Yield curves** affect forward prices and discounting
- **Inverted curves** (short > long) indicate recession expectations
- **Vol surfaces** capture smile/skew effects that flat vol ignores

In [ ]:
# =============================================================================
# STEP 1: Create Realistic Market Data
# =============================================================================

# Define market IDs (unique identifiers for market data)
spot_id = MarketId.parse("FX.SPOT.EURUSD")
vol_id = MarketId.parse("FX.VOL.EURUSD")
usd_curve_id = MarketId.parse("IR.ZERO.USD")
eur_curve_id = MarketId.parse("IR.ZERO.EUR")

# Current spot rate
SPOT = 1.0850

# Realistic USD zero rate curve (inverted - typical 2024+ environment)
usd_tenors = np.array([0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 20.0, 30.0])
usd_rates = np.array([0.0540, 0.0535, 0.0520, 0.0480, 0.0460, 0.0440, 0.0435, 0.0430, 0.0425, 0.0420])

# Realistic EUR zero rate curve (normal upward sloping)
eur_tenors = np.array([0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 20.0, 30.0])
eur_rates = np.array([0.0380, 0.0385, 0.0390, 0.0395, 0.0400, 0.0405, 0.0408, 0.0410, 0.0412, 0.0415])

# Realistic implied vol surface (expiry x strike grid)
vol_expiries = np.array([0.083, 0.167, 0.25, 0.5, 1.0])  # 1M, 2M, 3M, 6M, 1Y
vol_strikes = np.array([0.95, 0.98, 1.00, 1.02, 1.05, 1.08, 1.10, 1.12, 1.15]) * SPOT

# Vol grid with realistic smile (negative skew typical for FX)
vol_grid = np.array([
    # 1M: Higher short-dated vol, steep smile
    [0.115, 0.100, 0.088, 0.085, 0.090, 0.095, 0.100, 0.108, 0.118],
    # 2M
    [0.112, 0.098, 0.087, 0.084, 0.088, 0.093, 0.098, 0.105, 0.115],
    # 3M
    [0.110, 0.096, 0.086, 0.084, 0.087, 0.092, 0.096, 0.103, 0.112],
    # 6M: Lower vol, flatter smile
    [0.108, 0.095, 0.086, 0.085, 0.088, 0.092, 0.095, 0.100, 0.108],
    # 1Y: Lowest vol, flattest smile
    [0.105, 0.094, 0.088, 0.087, 0.090, 0.093, 0.096, 0.100, 0.106],
])

# Build the market object
market = Market(
    asof=date.today(),
    quotes={
        spot_id: Quote(value=SPOT),
    },
    curves={
        usd_curve_id: ZeroRateCurve(tenors=usd_tenors, zero_rates=usd_rates),
        eur_curve_id: ZeroRateCurve(tenors=eur_tenors, zero_rates=eur_rates),
    },
    vols={
        vol_id: GridVolSurface(
            expiries=vol_expiries,
            strikes=vol_strikes,
            implied_vols=vol_grid,
        ),
    },
)

print("Market Data Summary:")
print("="*60)
print(f"As of Date:      {market.asof}")
print(f"EURUSD Spot:     {SPOT:.4f}")
print(f"\nUSD Curve (inverted):")
print(f"  3M rate:       {usd_rates[0]:.2%}")
print(f"  1Y rate:       {usd_rates[2]:.2%}")
print(f"  10Y rate:      {usd_rates[7]:.2%}")
print(f"\nEUR Curve (normal):")
print(f"  3M rate:       {eur_rates[0]:.2%}")
print(f"  1Y rate:       {eur_rates[2]:.2%}")
print(f"  10Y rate:      {eur_rates[7]:.2%}")
print(f"\nVol Surface:")
print(f"  3M ATM vol:    {vol_grid[2, 4]:.2%}")
print(f"  1Y ATM vol:    {vol_grid[4, 4]:.2%}")
print(f"  Grid size:     {vol_grid.shape[0]} expiries x {vol_grid.shape[1]} strikes")

In [ ]:
# =============================================================================
# Visualize the yield curves and vol surface
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Yield curves
ax1 = axes[0]
ax1.plot(usd_tenors, usd_rates * 100, 'b-o', label='USD (inverted)', linewidth=2, markersize=5)
ax1.plot(eur_tenors, eur_rates * 100, 'r-s', label='EUR (normal)', linewidth=2, markersize=5)
ax1.set_xlabel('Tenor (years)')
ax1.set_ylabel('Zero Rate (%)')
ax1.set_title('Zero Rate Curves', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Vol smile at different expiries
ax2 = axes[1]
for i, exp in enumerate([0, 2, 4]):
    label = f"{vol_expiries[exp]*12:.0f}M" if vol_expiries[exp] < 1 else f"{vol_expiries[exp]:.0f}Y"
    ax2.plot(vol_strikes / SPOT, vol_grid[exp, :] * 100, '-o', label=label, linewidth=2, markersize=4)
ax2.axvline(x=1.0, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Strike / Spot (Moneyness)')
ax2.set_ylabel('Implied Vol (%)')
ax2.set_title('Vol Smile by Expiry', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Vol term structure
ax3 = axes[2]
atm_idx = len(vol_strikes) // 2
ax3.plot(vol_expiries, vol_grid[:, atm_idx] * 100, 'g-o', linewidth=2, markersize=6)
ax3.set_xlabel('Expiry (years)')
ax3.set_ylabel('ATM Implied Vol (%)')
ax3.set_title('ATM Vol Term Structure', fontweight='bold')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Creating and Pricing an Option

Now let's create an FX vanilla option and price it. The pricer will:
- Read the spot from market data
- Interpolate rates from the term structures at the option's expiry
- Interpolate vol from the surface at the option's expiry and strike

In [ ]:
# =============================================================================
# STEP 2: Create an Option Instrument
# =============================================================================

# Option parameters
STRIKE = 1.10          # Strike price (OTM call)
EXPIRY = 0.25          # 3 months (0.25 years)
NOTIONAL = 1_000_000   # EUR 1 million

# Create a EURUSD call option
call_option = FxVanillaEuropeanOption(
    option_type="call",
    notional=NOTIONAL,
    strike=STRIKE,
    expiry=EXPIRY,
    spot_id=spot_id,
    vol_id=vol_id,
    domestic_curve_id=usd_curve_id,
    foreign_curve_id=eur_curve_id,
)

# Create the corresponding put option
put_option = FxVanillaEuropeanOption(
    option_type="put",
    notional=NOTIONAL,
    strike=STRIKE,
    expiry=EXPIRY,
    spot_id=spot_id,
    vol_id=vol_id,
    domestic_curve_id=usd_curve_id,
    foreign_curve_id=eur_curve_id,
)

print("Option Details:")
print("="*50)
print(f"Option Type:     Call & Put")
print(f"Underlying:      EURUSD")
print(f"Current Spot:    {SPOT:.4f}")
print(f"Strike:          {STRIKE:.4f}")
print(f"Moneyness:       {SPOT/STRIKE:.2%} (Spot/Strike)")
print(f"Expiry:          {EXPIRY:.2f} years ({int(EXPIRY*365)} days)")
print(f"Notional:        EUR {NOTIONAL:,.0f}")

In [ ]:
# =============================================================================
# STEP 3: Price the Options
# =============================================================================

# Create a pricer
pricer = FxVanillaEuropeanOptionBsmPricer()

# Price the options (pricer reads from market directly)
call_pv = pricer.price(call_option, market)
put_pv = pricer.price(put_option, market)

# Per-unit prices (premium as % of notional)
call_premium_pct = call_pv / NOTIONAL * 100
put_premium_pct = put_pv / NOTIONAL * 100

print("\nPricing Results:")
print("="*50)
print(f"{'Option':<15} {'Premium (USD)':<20} {'% of Notional':<15}")
print("-"*50)
print(f"{'Call':<15} ${call_pv:>15,.2f} {call_premium_pct:>14.2f}%")
print(f"{'Put':<15} ${put_pv:>15,.2f} {put_premium_pct:>14.2f}%")
print("-"*50)

# What rates and vol were used?
usd_rate_at_expiry = np.interp(EXPIRY, usd_tenors, usd_rates)
eur_rate_at_expiry = np.interp(EXPIRY, eur_tenors, eur_rates)
vol_at_strike = market.vol_surface(vol_id).vol(expiry=EXPIRY, strike=STRIKE)

print(f"\nMarket data used at T={EXPIRY}Y:")
print(f"  USD rate:  {usd_rate_at_expiry:.2%}")
print(f"  EUR rate:  {eur_rate_at_expiry:.2%}")
print(f"  Impl vol:  {vol_at_strike:.2%}")

## 3. Computing Greeks

Greeks measure how the option price changes with respect to various risk factors:

| Greek | Formula | Meaning | Typical Range |
|-------|---------|---------|---------------|
| **Delta (Δ)** | ∂V/∂S | Change in value per unit spot move | -1 to +1 |
| **Gamma (Γ)** | ∂²V/∂S² | Rate of change of delta | Always positive |
| **Vega (ν)** | ∂V/∂σ | Change in value per 1% vol move | Always positive |
| **Theta (Θ)** | -∂V/∂t | Daily time decay | Usually negative |

In [ ]:
# =============================================================================
# STEP 4: Compute Greeks
# =============================================================================

# Compute Greeks
call_greeks = pricer.greeks(call_option, market)
put_greeks = pricer.greeks(put_option, market)

print("\nGreek Values (Notional = EUR 1,000,000):")
print("="*70)
print(f"{'Greek':<12} {'Call Value':<20} {'Put Value':<20} {'Interpretation'}")
print("-"*70)

# Delta
print(f"{'Delta':<12} {call_greeks['delta']:>15,.0f} USD {put_greeks['delta']:>15,.0f} USD    Per 0.0001 spot move")

# Gamma
print(f"{'Gamma':<12} {call_greeks['gamma']:>15,.0f}     {put_greeks['gamma']:>15,.0f}         Delta change per move")

# Vega
print(f"{'Vega':<12} {call_greeks['vega']:>15,.0f} USD {put_greeks['vega']:>15,.0f} USD    Per 1 vol point")

# Theta
print(f"{'Theta':<12} {call_greeks['theta']:>15,.0f} USD {put_greeks['theta']:>15,.0f} USD    Per day")

print("-"*70)

In [ ]:
# =============================================================================
# Greek Interpretation
# =============================================================================

print("\n" + "="*70)
print("Greek Interpretation for the EURUSD Call Option")
print("="*70)

print(f"\n1. DELTA = {call_greeks['delta']:,.0f} USD")
delta_pct = call_greeks['delta'] / (NOTIONAL * SPOT) * 100
print(f"   - This call is {'ITM' if SPOT > STRIKE else 'OTM'}, delta is ~{delta_pct:.1f}%")
print(f"   - If EURUSD moves up 100 pips (0.01), gain ~${call_greeks['delta'] * 0.01:,.0f}")

print(f"\n2. GAMMA = {call_greeks['gamma']:,.0f}")
print(f"   - Positive gamma: you benefit from large moves in either direction")
print(f"   - As spot moves, delta changes by this amount per unit spot move")

print(f"\n3. VEGA = ${call_greeks['vega']:,.0f} per 1 vol point")
print(f"   - If vol increases from {vol_at_strike:.0%} to {vol_at_strike+0.01:.0%}, gain ~${call_greeks['vega'] * 0.01:,.0f}")
print(f"   - This is a LONG volatility position")

print(f"\n4. THETA = ${call_greeks['theta']:,.0f} per day")
print(f"   - The option loses ~${abs(call_greeks['theta']):,.0f} in value each day")
print(f"   - Over {int(EXPIRY*365)} days, total decay ≈ premium paid")

## 4. Visualizing Option Payoffs

In [ ]:
# =============================================================================
# STEP 5: Visualize Option Payoffs
# =============================================================================

# Generate spot range for plotting
spot_range = np.linspace(SPOT * 0.85, SPOT * 1.15, 100)

# Calculate payoffs at expiry (intrinsic value)
call_payoff = np.maximum(spot_range - STRIKE, 0) * NOTIONAL
put_payoff = np.maximum(STRIKE - spot_range, 0) * NOTIONAL

# Calculate P&L (payoff minus premium)
call_pnl = call_payoff - call_pv
put_pnl = put_payoff - put_pv

# Create the plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Payoff at Expiry
ax1 = axes[0]
ax1.plot(spot_range, call_payoff / 1000, 'b-', linewidth=2, label='Call Payoff')
ax1.plot(spot_range, put_payoff / 1000, 'r-', linewidth=2, label='Put Payoff')
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax1.axvline(x=STRIKE, color='gray', linestyle='--', linewidth=1, label=f'Strike = {STRIKE}')
ax1.axvline(x=SPOT, color='green', linestyle=':', linewidth=1, label=f'Current Spot = {SPOT}')
ax1.set_xlabel('EURUSD Spot at Expiry', fontsize=12)
ax1.set_ylabel('Payoff (USD thousands)', fontsize=12)
ax1.set_title('Option Payoff at Expiry', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Plot 2: P&L Profile
ax2 = axes[1]
ax2.plot(spot_range, call_pnl / 1000, 'b-', linewidth=2, label='Call P&L')
ax2.plot(spot_range, put_pnl / 1000, 'r-', linewidth=2, label='Put P&L')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.axvline(x=STRIKE, color='gray', linestyle='--', linewidth=1)
ax2.axvline(x=SPOT, color='green', linestyle=':', linewidth=1)

# Shade profit/loss regions
ax2.fill_between(spot_range, call_pnl / 1000, 0, where=(call_pnl > 0), alpha=0.2, color='blue')
ax2.fill_between(spot_range, call_pnl / 1000, 0, where=(call_pnl < 0), alpha=0.2, color='red')

ax2.set_xlabel('EURUSD Spot at Expiry', fontsize=12)
ax2.set_ylabel('P&L (USD thousands)', fontsize=12)
ax2.set_title('Profit & Loss Profile (Including Premium)', fontsize=14, fontweight='bold')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print key levels
call_breakeven = STRIKE + call_pv / NOTIONAL
put_breakeven = STRIKE - put_pv / NOTIONAL

print(f"\nKey Levels:")
print(f"  Call breakeven: {call_breakeven:.4f}")
print(f"  Put breakeven:  {put_breakeven:.4f}")
print(f"  Call max loss:  ${call_pv:,.0f} (premium paid)")
print(f"  Put max loss:   ${put_pv:,.0f} (premium paid)")

## 5. Key Takeaways

### Realistic Market Data Matters
- **Term structures** affect forward prices and discounting differently at each expiry
- **Vol surfaces** capture the smile/skew that flat vol ignores
- Production systems always use full term structures, not flat assumptions

### Option Pricing
- Options have **time value** (premium above intrinsic value)
- Premium depends on moneyness, time, volatility, and rates

### Greeks
- **Delta** tells you directional exposure (hedge ratio)
- **Gamma** measures convexity (delta sensitivity)
- **Vega** measures vol exposure (long options = long vol)
- **Theta** is the cost of holding options (time decay)

In [ ]:
print("Try modifying the parameters above and re-running!")
print("\nSuggested experiments:")
print("  1. Change STRIKE to SPOT for an ATM option")
print("  2. Change EXPIRY to 1.0 for a 1-year option")
print("  3. Modify the vol surface to see how smile affects pricing")
print("  4. Compare results between inverted and normal USD curves")